In [1]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

BASE = Path("/content/drive/MyDrive/ArteryAI")

TEST_DIR = BASE / "Milestone-4" / "testing_reports"
TEST_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project connected successfully!")
print("Device:", device)
print("Testing directory:", TEST_DIR)


Mounted at /content/drive
Project connected successfully!
Device: cuda
Testing directory: /content/drive/MyDrive/ArteryAI/Milestone-4/testing_reports


###Load all three datasets

In [2]:

DATA_DIR = BASE / "data" / "processed" / "stenosis"

datasets = {}

for split in ["train", "val", "test"]:
    folder = DATA_DIR / split

    datasets[split] = {
        "images": np.load(folder / "images.npy"),
        "binary_masks": np.load(folder / "binary_masks.npy"),
        "multiclass_masks": np.load(folder / "multiclass_masks.npy"),
        "image_ids": np.load(folder / "image_ids.npy")
    }

    print(
        split.upper(),
        "Images:", datasets[split]["images"].shape,
        "Masks:", datasets[split]["binary_masks"].shape
    )

print("\nAll stenosis datasets loaded!")


TRAIN Images: (1000, 256, 256) Masks: (1000, 256, 256)
VAL Images: (200, 256, 256) Masks: (200, 256, 256)
TEST Images: (300, 256, 256) Masks: (300, 256, 256)

All stenosis datasets loaded!


###Load Stenosis U-Net architecture

In [3]:

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class StenosisUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(1, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        self.bottleneck = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec1 = DoubleConv(64, 32)

        self.output = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        b = self.bottleneck(self.pool(e3))

        d3 = self.dec3(
            torch.cat([self.up3(b), e3], dim=1)
        )
        d2 = self.dec2(
            torch.cat([self.up2(d3), e2], dim=1)
        )
        d1 = self.dec1(
            torch.cat([self.up1(d2), e1], dim=1)
        )

        return self.output(d1)


print("Stenosis U-Net architecture loaded!")


Stenosis U-Net architecture loaded!


###Load trained checkpoint

In [4]:

CHECKPOINT = BASE / "models" / "stenosis" / "stenosis_unet_v2.pth"

assert CHECKPOINT.exists(), (
    f"Checkpoint not found: {CHECKPOINT}"
)

stenosis_model = StenosisUNet().to(device)

state = torch.load(
    CHECKPOINT,
    map_location=device,
    weights_only=True
)

stenosis_model.load_state_dict(state)
stenosis_model.eval()

BEST_THRESHOLD = 0.80

print("Stenosis model loaded successfully!")
print("Checkpoint:", CHECKPOINT)
print("Threshold:", BEST_THRESHOLD)


Stenosis model loaded successfully!
Checkpoint: /content/drive/MyDrive/ArteryAI/models/stenosis/stenosis_unet_v2.pth
Threshold: 0.8


###Prepare Test Data and Run First Test

In [5]:

class StenosisTestDataset(Dataset):
    def __init__(self, split):
        self.images = datasets[split]["images"]
        self.masks = datasets[split]["binary_masks"]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].astype(np.float32)
        mask = self.masks[idx].astype(np.float32)

        image = (image - image.min()) / (
            image.max() - image.min() + 1e-8
        )

        return (
            torch.tensor(image).unsqueeze(0),
            torch.tensor(mask).unsqueeze(0)
        )


test_loader = DataLoader(
    StenosisTestDataset("test"),
    batch_size=8,
    shuffle=False
)

print("Test samples:", len(test_loader.dataset))
print("Test batches:", len(test_loader))


Test samples: 300
Test batches: 38


###First End-to-End Test

In [6]:

stenosis_model.eval()

images, masks = next(iter(test_loader))

images = images.to(device)
masks = masks.to(device)

start = time.perf_counter()

with torch.inference_mode():
    logits = stenosis_model(images)
    probabilities = torch.sigmoid(logits)
    predictions = (
        probabilities >= BEST_THRESHOLD
    ).float()

if device.type == "cuda":
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start

print("INPUT:", images.shape)
print("GROUND TRUTH:", masks.shape)
print("PREDICTION:", predictions.shape)

print("Prediction range:",
      probabilities.min().item(),
      probabilities.max().item())

print("Inference time:", round(elapsed, 4), "seconds")

assert predictions.shape == masks.shape
assert torch.isfinite(probabilities).all()
assert predictions.min() >= 0
assert predictions.max() <= 1

print("\nFIRST END-TO-END TEST PASSED!")


INPUT: torch.Size([8, 1, 256, 256])
GROUND TRUTH: torch.Size([8, 1, 256, 256])
PREDICTION: torch.Size([8, 1, 256, 256])
Prediction range: 5.433361138784676e-07 1.0
Inference time: 1.6775 seconds

FIRST END-TO-END TEST PASSED!


###Complete End-to-End Testing (300 Images)

In [7]:

import time
import json
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

stenosis_model.eval()

TP = FP = TN = FN = 0
total_time = 0
total_images = 0
batch_results = []

for batch_idx, (images, masks) in enumerate(
    tqdm(test_loader, desc="End-to-End Testing")
):

    images = images.to(device)
    masks = masks.to(device)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        logits = stenosis_model(images)
        probabilities = torch.sigmoid(logits)
        predictions = (
            probabilities >= BEST_THRESHOLD
        ).float()

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    total_time += elapsed
    total_images += images.size(0)

    TP += ((predictions == 1) & (masks == 1)).sum().item()
    FP += ((predictions == 1) & (masks == 0)).sum().item()
    TN += ((predictions == 0) & (masks == 0)).sum().item()
    FN += ((predictions == 0) & (masks == 1)).sum().item()

    batch_results.append({
        "batch": batch_idx + 1,
        "images": images.size(0),
        "inference_time_seconds": elapsed,
        "prediction_shape_valid": (
            predictions.shape == masks.shape
        ),
        "finite_predictions": torch.isfinite(
            probabilities
        ).all().item()
    })

# Calculate metrics
epsilon = 1e-8

dice = (2 * TP) / (2 * TP + FP + FN + epsilon)
iou = TP / (TP + FP + FN + epsilon)
precision = TP / (TP + FP + epsilon)
recall = TP / (TP + FN + epsilon)
specificity = TN / (TN + FP + epsilon)
accuracy = (TP + TN) / (TP + TN + FP + FN + epsilon)

f1 = 2 * precision * recall / (precision + recall + epsilon)

results = {
    "total_test_images": total_images,
    "threshold": BEST_THRESHOLD,
    "TP": int(TP),
    "FP": int(FP),
    "TN": int(TN),
    "FN": int(FN),
    "Dice": dice,
    "IoU": iou,
    "Precision": precision,
    "Recall": recall,
    "F1_Score": f1,
    "Sensitivity": recall,
    "Specificity": specificity,
    "Accuracy": accuracy,
    "Total_Inference_Time": total_time,
    "Average_Time_Per_Image": total_time / total_images,
    "Images_Per_Second": total_images / total_time
}

# Save reports
with open(TEST_DIR / "end_to_end_test.json", "w") as f:
    json.dump(results, f, indent=4)

pd.DataFrame(batch_results).to_csv(
    TEST_DIR / "batch_test_results.csv",
    index=False
)

pd.DataFrame([results]).to_csv(
    TEST_DIR / "end_to_end_metrics.csv",
    index=False
)

print("\nEND-TO-END TESTING COMPLETED")
print("=" * 45)

for key, value in results.items():
    print(f"{key}: {value}")

print("\nReports saved at:", TEST_DIR)


End-to-End Testing:   0%|          | 0/38 [00:00<?, ?it/s]


END-TO-END TESTING COMPLETED
total_test_images: 300
threshold: 0.8
TP: 106055
FP: 120632
TN: 19351511
FN: 82602
Dice: 0.5106851188412374
IoU: 0.34289935949871014
Precision: 0.467847737188261
Recall: 0.5621577784020437
F1_Score: 0.5106851138831436
Sensitivity: 0.5621577784020437
Specificity: 0.9938048934829612
Accuracy: 0.989662984212239
Total_Inference_Time: 1.7832022499993627
Average_Time_Per_Image: 0.005944007499997876
Images_Per_Second: 168.23666524652893

Reports saved at: /content/drive/MyDrive/ArteryAI/Milestone-4/testing_reports


###Performance Testing

In [8]:

import time
import pandas as pd
import torch

stenosis_model.eval()

performance_results = []

# Warm-up
warmup = torch.randn(1, 1, 256, 256).to(device)

with torch.inference_mode():
    for _ in range(5):
        stenosis_model(warmup)

if device.type == "cuda":
    torch.cuda.synchronize()

# Benchmark
for batch_size in [1, 4, 8, 16]:

    loader = DataLoader(
        StenosisTestDataset("test"),
        batch_size=batch_size,
        shuffle=False
    )

    start = time.perf_counter()
    count = 0

    with torch.inference_mode():
        for images, masks in loader:
            images = images.to(device)
            output = stenosis_model(images)
            count += images.size(0)

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    performance_results.append({
        "batch_size": batch_size,
        "total_images": count,
        "total_time_seconds": elapsed,
        "average_time_per_image": elapsed / count,
        "images_per_second": count / elapsed
    })

performance_df = pd.DataFrame(performance_results)

display(performance_df)

performance_df.to_csv(
    TEST_DIR / "performance_test.csv",
    index=False
)

print("Performance testing completed!")


,batch_size,total_images,total_time_seconds,average_time_per_image,images_per_second
0,1,300,1.710320,0.005701,175.405813
1,4,300,1.659283,0.005531,180.800937
2,8,300,1.815785,0.006053,165.217780
3,16,300,1.561955,0.005207,192.067046


Performance testing completed!


###Security Testing

In [9]:

import torch
import numpy as np
import hashlib

security_results = []

def security_test(name, passed, details):
    security_results.append({
        "Test": name,
        "Status": "PASS" if passed else "FAIL",
        "Details": details
    })

# Test 1: Invalid image shape
try:
    invalid = torch.randn(1, 1, 128, 128).to(device)

    with torch.inference_mode():
        stenosis_model(invalid)

    security_test(
        "Invalid image dimensions",
        False,
        "Model accepted unexpected dimensions"
    )

except Exception:
    security_test(
        "Invalid image dimensions",
        True,
        "Invalid input rejected"
    )

# Test 2: NaN input
try:
    invalid = torch.full(
        (1, 1, 256, 256),
        float("nan")
    ).to(device)

    with torch.inference_mode():
        output = stenosis_model(invalid)

    passed = not torch.isfinite(output).all().item()

    security_test(
        "NaN input handling",
        passed,
        "Non-finite input detected in output; production validation required"
    )

except Exception:
    security_test(
        "NaN input handling",
        True,
        "Invalid input rejected"
    )

# Test 3: Checkpoint integrity
checkpoint_hash = hashlib.sha256(
    CHECKPOINT.read_bytes()
).hexdigest()

security_test(
    "Checkpoint integrity",
    len(checkpoint_hash) == 64,
    f"SHA256: {checkpoint_hash}"
)

# Test 4: Output directory
security_test(
    "Report directory access",
    TEST_DIR.exists() and os.access(TEST_DIR, os.W_OK),
    "Testing output directory is writable"
)

security_df = pd.DataFrame(security_results)

display(security_df)

security_df.to_csv(
    TEST_DIR / "security_test.csv",
    index=False
)

print("Security testing completed!")


,Test,Status,Details
0,Invalid image dimensions,FAIL,Model accepted unexpected dimensions
1,NaN input handling,PASS,Non-finite input detected in output; productio...
2,Checkpoint integrity,PASS,SHA256: 8244583c28866d0aab1a73f7ba90cefbe2ed31...
3,Report directory access,PASS,Testing output directory is writable


Security testing completed!


In [10]:

import torch
import pandas as pd
import hashlib

def safe_predict(image, model, device):
    # Check input type
    if not isinstance(image, torch.Tensor):
        raise TypeError("Input must be a torch Tensor")

    # Check dimensions
    if image.ndim != 4:
        raise ValueError("Expected [B, 1, 256, 256]")

    if image.shape[1:] != (1, 256, 256):
        raise ValueError("Invalid image dimensions")

    # Check NaN and Inf
    if not torch.isfinite(image).all():
        raise ValueError("NaN or Inf detected in input")

    if image.min() < 0 or image.max() > 1:
        raise ValueError("Input must be normalized to [0,1]")

    model.eval()

    with torch.inference_mode():
        output = model(image.to(device))

        if not torch.isfinite(output).all():
            raise ValueError("Non-finite model output detected")

        return torch.sigmoid(output)


security_results = []

def run_security_test(name, function):
    try:
        function()
        status = "FAIL"
        details = "Invalid input was not rejected"
    except (ValueError, TypeError) as e:
        status = "PASS"
        details = str(e)
    except Exception as e:
        status = "ERROR"
        details = str(e)

    security_results.append({
        "Test": name,
        "Status": status,
        "Details": details
    })

# Valid input should work
valid = torch.rand(1, 1, 256, 256)

try:
    result = safe_predict(valid, stenosis_model, device)
    security_results.append({
        "Test": "Valid input acceptance",
        "Status": "PASS",
        "Details": "Valid input processed successfully"
    })
except Exception as e:
    security_results.append({
        "Test": "Valid input acceptance",
        "Status": "FAIL",
        "Details": str(e)
    })

# Invalid dimensions
run_security_test(
    "Invalid image dimensions",
    lambda: safe_predict(
        torch.rand(1, 1, 128, 128),
        stenosis_model, device
    )
)

# NaN input
nan_image = torch.rand(1, 1, 256, 256)
nan_image[0, 0, 0, 0] = float("nan")

run_security_test(
    "NaN input rejection",
    lambda: safe_predict(
        nan_image, stenosis_model, device
    )
)

# Inf input
inf_image = torch.rand(1, 1, 256, 256)
inf_image[0, 0, 0, 0] = float("inf")

run_security_test(
    "Infinite input rejection",
    lambda: safe_predict(
        inf_image, stenosis_model, device
    )
)

security_df = pd.DataFrame(security_results)
display(security_df)

security_df.to_csv(
    TEST_DIR / "security_test_fixed.csv",
    index=False
)

print("Input validation security testing completed!")


,Test,Status,Details
0,Valid input acceptance,PASS,Valid input processed successfully
1,Invalid image dimensions,PASS,Invalid image dimensions
2,NaN input rejection,PASS,NaN or Inf detected in input
3,Infinite input rejection,PASS,NaN or Inf detected in input


Input validation security testing completed!


In [11]:

import json
import pandas as pd
from pathlib import Path

report_dir = TEST_DIR

# Load End-to-End results
with open(report_dir / "end_to_end_test.json") as f:
    e2e = json.load(f)

# Load Performance results
performance = pd.read_csv(
    report_dir / "performance_test.csv"
)

# Load corrected Security results
security = pd.read_csv(
    report_dir / "security_test_fixed.csv"
)

final_report = {
    "project": "Artery Segmentation and Stenosis Detection",
    "testing_date": pd.Timestamp.now().isoformat(),
    "end_to_end_testing": e2e,
    "performance_testing": performance.to_dict(
        orient="records"
    ),
    "security_testing": security.to_dict(
        orient="records"
    )
}

with open(report_dir / "final_testing_report.json", "w") as f:
    json.dump(final_report, f, indent=4)

summary = pd.DataFrame([
    ["End-to-End Testing", "Completed", "300 images"],
    ["Performance Testing", "Completed", "4 batch sizes"],
    ["Security Input Validation", "Completed", "4 tests passed"],
    ["API Testing", "Not verified", "Backend verification required"]
], columns=["Testing Module", "Status", "Details"])

summary.to_csv(
    report_dir / "final_testing_summary.csv",
    index=False
)

display(summary)

print("\nFinal testing reports saved!")
print(report_dir)


,Testing Module,Status,Details
0,End-to-End Testing,Completed,300 images
1,Performance Testing,Completed,4 batch sizes
2,Security Input Validation,Completed,4 tests passed
3,API Testing,Not verified,Backend verification required



Final testing reports saved!
/content/drive/MyDrive/ArteryAI/Milestone-4/testing_reports
